In [ ]:
import argparse
import json
import re
import os
from tqdm import tqdm
from PIL import Image
from pathlib import Path

# --- 您的配置 ---
MODEL_NAME = "Dolphin_v2"
INPUT_FILE = "MPDocBench.json" 
OUTPUT_PATH = f"./markdown/{MODEL_NAME}"

MD_PATH = OUTPUT_PATH + "_md"
os.makedirs(MD_PATH, exist_ok=True)
MD_PATH = Path(MD_PATH)

# --- 数据加载 ---
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    raw_data = {}
    for item in data:
        page_info = item["page_info"]
        images_list = page_info["images_list"]
        annotations_list = page_info["annotations_list"]
        image_path = page_info["image_path"]
        pdf_name = os.path.splitext(image_path)[0]
        if pdf_name not in raw_data:
            raw_data[pdf_name] = []
            page_id = 0
            for img, ann in zip(images_list, annotations_list):
                raw_data[pdf_name].append((page_id, img, ann))
                page_id += 1
        else:
            print(f"Warning: duplicate pdf_name {pdf_name} found. Skipping.")
            continue
    data = raw_data

# 使用tqdm来显示处理进度
for pdf_name in tqdm(data.keys(), desc="Processing PDFs"):
    pdf_dir = Path(os.path.join(OUTPUT_PATH, pdf_name))
    md_path = pdf_dir / "markdown" / f"{pdf_name}.md"
    json_path = pdf_dir / "recognition_json" / f"{pdf_name}.json"
    md_final_path = MD_PATH / f"{pdf_name}.md"

    if not md_path.exists() or not json_path.exists():
        print(f"\n[WARN] 跳过 {pdf_name}，因为缺少 markdown 或 json 文件。")
        continue

    md_content = md_path.read_text(encoding="utf-8")
    json_data = json.load(open(json_path))

    # 1. 预处理：收集所有需要替换的信息
    info_map = {}
    page_to_size = {}

    pages = json_data.get("pages", [])
    for page in pages:
        image_path = page.get("image_path")
        page_number = page.get("page_number")
        
        if page_number not in page_to_size and image_path:
            try:
                with Image.open(image_path) as img:
                    width, height = img.size
                    page_to_size[page_number] = (width, height)
            except FileNotFoundError:
                print(f"\n[WARN] 图片文件未找到: {image_path}")
                continue

        elements = page.get("elements", [])
        for element in elements:
            if (element.get("label") == "fig" and 
                element.get("text") and 
                element.get("figure_path")):
                
                text_to_replace = element["text"]
                info_map[text_to_replace] = {
                    "page_number": page_number,
                    "bbox": element["bbox"],
                }

    if not info_map:
        md_final_path.write_text(md_content, encoding="utf-8")
        continue

    # 2. 定义带有提示的回调函数
    def replace_figure_path(match):
        matched_text = match.group(0)
        
        # 尝试在预处理的映射中找到匹配的文本
        if matched_text in info_map:
            info = info_map[matched_text]
            page_number = info["page_number"]
            
            # 检查该页面的尺寸是否已知
            if page_number in page_to_size:
                x0, y0, x1, y1 = info["bbox"]
                width, height = page_to_size[page_number]
                
                # 计算归一化坐标和新文件名
                norm_x0 = round((x0 / width) * 1000)
                norm_y0 = round((y0 / height) * 1000)
                norm_x1 = round((x1 / width) * 1000)
                norm_y1 = round((y1 / height) * 1000)
                
                final_filename = f"page{page_number}_{norm_x0}_{norm_y0}_{norm_x1}_{norm_y1}.jpg"
                alt_text = ""
                
                return f"![{alt_text}]({final_filename})"
            else:
                print(f"\n[WARN] 在 {pdf_name} 中，找到图片 '{matched_text}' 的信息，但未找到其所在页面 {page_number} 的尺寸信息，将不进行替换。")

        else:
            print(f"\n[WARN] 在 {pdf_name} 中，Markdown中的图片 '{matched_text}' 在JSON数据中未找到对应条目，将不进行替换。")
        return matched_text

    # 3. 构建正则表达式并执行替换
    pattern = '|'.join(re.escape(key) for key in info_map.keys())
    new_md_content = re.sub(pattern, replace_figure_path, md_content)
    md_final_path.write_text(new_md_content, encoding="utf-8")